# Generalised Additive Models with pyGAM

## Overview

GAMs extend GLMs by replacing linear predictors with flexible smooth functions of covariates. They capture non-linear relationships while remaining interpretable — each smooth term can be plotted and understood independently.

**GAM formula:**
```
g(E[Y]) = β₀ + f₁(x₁) + f₂(x₂) + ... + βₖzₖ
```
where `f()` are penalised spline smooth functions and `g()` is a link function.

**pyGAM vs R's mgcv:**

| Feature | pyGAM | mgcv (R) |
|---|---|---|
| Smooth types | B-splines | B-splines, thin-plate, cubic |
| Family support | Gaussian, Binomial, Poisson, Gamma | Full GLM families + more |
| Automatic smoothing | GCV/REML | GCV/REML |
| Tensor products | Limited | Full |
| Production use | Moderate | Gold standard |

**Honest note:** pyGAM is capable for standard use cases but is less mature than R's mgcv. For complex GAMMs or ecological community data, R remains the better tool.

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(42)
n = 300
elevation = rng.uniform(50, 400, n)
nitrate   = rng.gamma(2, 2, n)
temp      = 15 - 0.02*elevation + rng.normal(0, 1.5, n)
treatment = rng.choice([0, 1], n)
# True non-linear relationship with elevation (hump-shaped)
richness  = (5 + 20*np.exp(-((elevation-200)**2)/(2*80**2))
             - 0.8*nitrate + 2.5*treatment
             + rng.normal(0, 2.5, n)).clip(0)
df = pd.DataFrame(dict(elevation=elevation, nitrate=nitrate,
                        temp=temp, treatment=treatment, richness=richness))
print(f"n={n}, richness range: {richness.min():.1f}–{richness.max():.1f}")

---
## Fitting a GAM

In [ ]:
try:
    from pygam import LinearGAM, LogisticGAM, PoissonGAM, s, f, l
    X = df[['elevation','nitrate','treatment']].values
    y = df['richness'].values
    # s() = smooth term, f() = factor/categorical, l() = linear
    gam = LinearGAM(s(0) + s(1) + l(2)).fit(X, y)
    print(gam.summary())
    print(f"\nGAM R-squared: {gam.statistics_['pseudo_r2']['explained_deviance']:.3f}")
except ImportError:
    print("pygam not installed: pip install pygam")
    print("GAM syntax: LinearGAM(s(0) + s(1) + l(2)).fit(X, y)")
    print("  s(i) = smooth of column i")
    print("  l(i) = linear term for column i")
    print("  f(i) = factor (categorical) term")

---
## Smooth Term Visualisation

In [ ]:
try:
    from pygam import LinearGAM, s, l
    X = df[['elevation','nitrate','treatment']].values
    y = df['richness'].values
    gam = LinearGAM(s(0) + s(1) + l(2)).fit(X, y)
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    titles = ['Elevation (smooth)', 'Nitrate (smooth)', 'Treatment (linear)']
    for i, (ax, title) in enumerate(zip(axes, titles)):
        XX = gam.generate_X_grid(term=i)
        pdep, confi = gam.partial_dependence(term=i, X=XX, width=0.95)
        ax.plot(XX[:, i], pdep, color='steelblue', lw=2)
        ax.fill_between(XX[:, i], confi[:, 0], confi[:, 1],
                        alpha=0.3, color='steelblue')
        ax.axhline(0, color='grey', lw=0.8, linestyle='--')
        ax.set_xlabel(title.split(' ')[0])
        ax.set_ylabel('Partial effect')
        ax.set_title(title)
    plt.suptitle('GAM Partial Effects (95% CI)', y=1.01)
    plt.tight_layout(); plt.show()
except ImportError:
    print("pip install pygam to generate partial dependence plots")
    print("Each smooth shows the marginal effect of one predictor")
    print("Flat region = no effect; curve = non-linear relationship")

---
## Smoothing Parameter Selection

In [ ]:
try:
    from pygam import LinearGAM, s, l
    X = df[['elevation','nitrate','treatment']].values
    y = df['richness'].values
    # Grid search over smoothing lambdas
    gam_tuned = LinearGAM(s(0) + s(1) + l(2))
    lam_grid = np.logspace(-3, 3, 20)
    gam_tuned.gridsearch(X, y, lam=lam_grid, progress=False)
    print(f"Optimal lambda: {gam_tuned.lam}")
    print(f"GCV score:      {gam_tuned.statistics_['GCV']:.4f}")
    # Compare: oversmoothed, auto, undersmoothed
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    for ax, lam, title in zip(axes, [1e4, gam_tuned.lam[0], 1e-4],
                                ['Oversmoothed (λ=1e4)',
                                 f'Optimal (λ={gam_tuned.lam[0]:.2f})',
                                 'Undersmoothed (λ=1e-4)']):
        g = LinearGAM(s(0, lam=lam) + s(1) + l(2)).fit(X, y)
        XX = g.generate_X_grid(term=0)
        pd_, ci = g.partial_dependence(term=0, X=XX, width=0.95)
        ax.scatter(X[:,0], y - y.mean(), alpha=0.2, s=10, color='steelblue')
        ax.plot(XX[:,0], pd_, color='#e74c3c', lw=2)
        ax.set_title(title)
        ax.set_xlabel('Elevation')
    plt.tight_layout(); plt.show()
except ImportError:
    print("pip install pygam")

In [ ]:
try:
    from pygam import LogisticGAM, PoissonGAM, s, l
    # Logistic GAM for binary outcome
    restored = (df['richness'] > df['richness'].mean()).astype(int).values
    X = df[['elevation','nitrate','treatment']].values
    log_gam = LogisticGAM(s(0) + s(1) + l(2)).fit(X, restored)
    print(f"Logistic GAM pseudo-R2: {log_gam.statistics_['pseudo_r2']['explained_deviance']:.3f}")
    # Poisson GAM for counts
    counts = rng.poisson(np.clip(df['richness']/2, 0.5, None))
    pois_gam = PoissonGAM(s(0) + s(1) + l(2)).fit(X, counts)
    print(f"Poisson GAM pseudo-R2: {pois_gam.statistics_['pseudo_r2']['explained_deviance']:.3f}")
    print("\nGAM family selection mirrors GLM family selection:")
    print("  Binary -> LogisticGAM")
    print("  Count  -> PoissonGAM")
    print("  Continuous positive -> GammaGAM")
except ImportError:
    print("pip install pygam")

---

## Common Pitfalls

**1. Using GAMs when a linear term suffices**  
Smooth terms have more degrees of freedom than linear terms. If the relationship is genuinely linear, a GAM smooth will produce a nearly-straight line but with wider confidence intervals than a simple linear fit. Always check whether the smooth is distinguishable from a straight line before preferring a GAM over a GLM.

**2. Not visualising smooth terms after fitting**  
The summary table for a GAM reports an approximate significance test for each smooth, but this says nothing about the shape of the effect. Always plot all smooth terms — a significant smooth that is actually nearly linear, or that reverses direction in a scientifically implausible way, may indicate a data problem rather than a true non-linearity.

**3. Extrapolating GAM predictions beyond the training data range**  
Spline smooths are poorly constrained outside the range of training data and can produce extreme, implausible predictions. Always restrict predictions to the observed covariate range and flag any extrapolation explicitly.

**4. Treating pyGAM as equivalent to R's mgcv for complex models**  
pyGAM lacks tensor product smooths, GAMM (random effects + GAM), and some GLM families available in mgcv. For ecologically complex models with spatial smooths, nested random effects, and non-standard distributions, R's mgcv is substantially more capable.

**5. Ignoring concurvity (the GAM equivalent of multicollinearity)**  
Concurvity occurs when one smooth can be approximated as a function of other smooths. High concurvity inflates standard errors on affected terms, similar to multicollinearity in linear models. pyGAM does not diagnose concurvity directly — check for it by examining how estimates change when correlated predictors are added or removed.
---
*python_methods_library - Samantha McGarrigle*